In [ ]:

import sys
import pandas as pd
import numpy as np
import torch
import importlib
import json
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from tqdm import tqdm
import logging
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.offsetbox import AnchoredOffsetbox, TextArea, VPacker, HPacker, DrawingArea
from matplotlib.patches import Rectangle

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    roc_curve,
    precision_recall_curve,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/dev/notebooks/simple_model_testing")
DATA_DIR = PROJECT_DIR / "data"
CHKPT_DIR = PROJECT_DIR / "checkpoints"
CHKPT_COPY_DIR = PROJECT_DIR / "checkpoints copy"
RESULT_DIR = PROJECT_DIR / "testing_results"

sys.path.append(str(PROJECT_DIR))

import models.tf_to_tg as tf_to_tg_module
import models.tf_to_dna as tf_to_dna_module
import scripts.build_tf_to_tg_train_data as tf_tg_data_builder
import plotting_utils
import utils
import warnings

warnings.filterwarnings(
    "ignore",
    message="You are using `torch.load` with `weights_only=False`.*",
    category=FutureWarning,
)

tf_tg_input_cache_dir = DATA_DIR / "tf_tg_training_cache"

all_evaluation_plot_dir = PROJECT_DIR / "plots" / "model_vs_test_set_evaluation_figs"
all_evaluation_plot_dir.mkdir(exist_ok=True)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

In [99]:
def find_latest_checkpoint(cell_type, sample_name, training_number=None) -> Path:
    sample_chkpt_dir = CHKPT_DIR / cell_type / sample_name
    
    if not sample_chkpt_dir.exists():
        logging.warning(f"No checkpoints found for {cell_type} {sample_name} in {sample_chkpt_dir}")
        return None
    
    if training_number is not None:
        slurm_job_dirs = [d for d in sample_chkpt_dir.iterdir() if d.is_dir() and d.name.startswith(f"tf_tg_train_{sample_name}_{training_number}")]
    else:
        slurm_job_dirs = [d for d in sample_chkpt_dir.iterdir() if d.is_dir() and d.name.startswith(f"tf_tg_train_{sample_name}_")]
    
    if not slurm_job_dirs:
        logging.warning(f"No checkpoint directories found for {cell_type} {sample_name} in {sample_chkpt_dir}")
        return None
    
    latest_chkpt_dir = max(slurm_job_dirs, key=lambda d: int(d.name.split("_")[-1]))
    slurm_job_id = latest_chkpt_dir.name.split("_")[-1]
    
    chkpt_files = list(latest_chkpt_dir.glob("epoch=*-val_auroc=*-val_loss=*.ckpt"))
    if not chkpt_files:
        logging.warning(f"No checkpoint files found for {sample_name} in {latest_chkpt_dir}")
        return None
    
    latest_chkpt_file = max(chkpt_files, key=lambda f: int(f.stem.split("-")[0].split("=")[1]))
    epoch = latest_chkpt_file.stem.split("-")[0].split("=")[1]
    
    logging.info(f"Latest checkpoint for {cell_type} {sample_name}: Job {slurm_job_id} Epoch {epoch}")
    return latest_chkpt_file

def load_tf_tg_regulation_model(
    tf_dna_model_path: Path, 
    tf_tg_model_path: Path,
    tf_embeddings_tensor: torch.Tensor,
    tf_mask_tensor: torch.Tensor
    ) -> tf_to_tg_module.TFTGRegulationModel:
    
    # 1) Recreate the base TF→DNA model with the same hyperparameters
    base_model = tf_to_dna_module.TFPeakBindingModel(
        tf_embedding_dim=128,
        hidden_dim=128,
        dropout=0.3,
        num_layers=4,
        num_heads=4,
        dim_head=32,
    )

    # 2) Wrap in Lightning module and load checkpoint
    lit_model = tf_to_dna_module.LitTFPeakBindingModel.load_from_checkpoint(
        checkpoint_path=tf_dna_model_path,
        model=base_model,
        tf_embeddings_tensor=tf_embeddings_tensor,
        tf_mask_tensor=tf_mask_tensor,
        lr=1e-4,
        weight_decay=1e-4,
        pos_weight=None,
    )

    # 4) Get the trained TF-DNA model and freeze it
    trained_tf_peak_model = lit_model.model
    trained_tf_peak_model.eval()
    for p in trained_tf_peak_model.parameters():
        p.requires_grad = False

    # 5) Create the TF-TG model object using the trained TF-DNA model, and load the trained model checkpoint
    tf_tg_model = tf_to_tg_module.LitTFTGRegulationModel.load_from_checkpoint(
        checkpoint_path=tf_tg_model_path,
        model=tf_to_tg_module.TFTGRegulationModel(
            pretrained_tf_peak_model=trained_tf_peak_model,
            d_model=128,
            tf_peak_chunk_size=256,
        ),
        lr=1e-4,
        weight_decay=1e-4,
        pos_weight=None,
    )
    
    return tf_tg_model

@torch.no_grad()
def move_batch_to_device(batch, device):
    moved = {
        "tf_embedding": batch["tf_embedding"].to(device, non_blocking=True),
        "tf_mask": batch["tf_mask"].to(device, non_blocking=True),
        "peak_sequences": batch["peak_sequences"].to(device, non_blocking=True),
        "peak_accessibility": batch["peak_accessibility"].to(device, non_blocking=True),
        "peak_distance": batch["peak_distance"].to(device, non_blocking=True),
        "tf_expression": batch["tf_expression"].to(device, non_blocking=True),
        "tg_expression": batch["tg_expression"].to(device, non_blocking=True),
    }

    if "cell_mask" in batch:
        moved["cell_mask"] = batch["cell_mask"].to(device, non_blocking=True)

    if "peak_mask" in batch:
        moved["peak_mask"] = batch["peak_mask"].to(device, non_blocking=True)

    return moved

def generate_model_predictions(model, data_loader, device, tf_idx_to_name, tg_idx_to_name):
    pooling_mode = "lse"
    pooling_temperature = 1.0

    model = model.to(device)
    model.eval()
    
    if device.type == "cuda":
        model = torch.compile(model, mode="reduce-overhead")

    tf_indices_list = []
    tg_indices_list = []
    all_scores = []

    with torch.inference_mode():
        for batch in tqdm(data_loader, desc="Evaluating", ncols=100):
            tf_indices = batch["tf_idx"].detach().cpu().numpy().ravel()
            tg_indices = batch["tg_idx"].detach().cpu().numpy().ravel()

            batch = move_batch_to_device(batch, device)

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(device.type == "cuda")):
                edge_logits, _ = model(
                    tf_embedding=batch["tf_embedding"],
                    tf_mask=batch["tf_mask"],
                    peak_sequences=batch["peak_sequences"],
                    peak_accessibility=batch["peak_accessibility"],
                    peak_distance=batch["peak_distance"],
                    tf_expression=batch["tf_expression"],
                    tg_expression=batch["tg_expression"],
                    peak_mask=batch.get("peak_mask", None),
                    cell_mask=batch["cell_mask"],
                    pooling_mode=pooling_mode,
                    pooling_temperature=pooling_temperature,
                )

            scores = torch.sigmoid(edge_logits.float())

            tf_indices_list.append(tf_indices)
            tg_indices_list.append(tg_indices)
            all_scores.append(scores.detach().cpu().numpy().ravel())

    all_tf_indices_flat = np.concatenate(tf_indices_list)
    all_tg_indices_flat = np.concatenate(tg_indices_list)
    all_scores_flat = np.concatenate(all_scores)

    tf_names = [tf_idx_to_name[int(idx)] for idx in all_tf_indices_flat]
    tg_names = [tg_idx_to_name[int(idx)] for idx in all_tg_indices_flat]

    prediction_df = pd.DataFrame({
        "Source": tf_names,
        "Target": tg_names,
        "Score": all_scores_flat,
    })

    prediction_df = (
        prediction_df.groupby(["Source", "Target"], as_index=False)["Score"]
        .median()
    )

    return prediction_df

def create_tf_tg_index_to_name_mappings(metadata):
    tf_idx_to_name = {idx: name for name, idx in metadata["tf_name_to_idx"].items()}
    tg_idx_to_name = {idx: name for name, idx in metadata["tg_id_to_idx"].items()}
    return tf_idx_to_name, tg_idx_to_name


In [3]:
mm10_tf_dna_path = CHKPT_DIR / "tf_dna_mm10_3671604" / "epoch=08-val_auroc=0.9177-val_loss=0.2783.ckpt"
hg38_tf_dna_path = CHKPT_DIR / "tf_dna_hg38_3683606" / "epoch=13-val_auroc=0.9566-val_loss=0.2042.ckpt"

tf_dna_model_checkpoints = {
    "mESC": mm10_tf_dna_path,
    "iPSC": hg38_tf_dna_path,
    "Macrophage": hg38_tf_dna_path,
    "K562": hg38_tf_dna_path
}

tf_tg_model_checkpoints = {
    "mESC": {
        "E7.5_rep1": CHKPT_DIR / "mESC" / "E7.5_rep1" / "tf_tg_train_E7.5_rep1_3675131" / "epoch_11_best_model.ckpt",
        "E7.5_rep2": find_latest_checkpoint("mESC", "E7.5_rep2"),
        "E8.5_rep1": find_latest_checkpoint("mESC", "E8.5_rep1", training_number="3691937"),
        "E8.5_rep2": find_latest_checkpoint("mESC", "E8.5_rep2"),
    },
    "iPSC": {
        "WT_D13_rep1": find_latest_checkpoint("iPSC", "WT_D13_rep1"),
    },
    "Macrophage": {
        "buffer_1": find_latest_checkpoint("Macrophage", "buffer_1", training_number="3685893"),
        "buffer_2": find_latest_checkpoint("Macrophage", "buffer_2"),
        "buffer_3": find_latest_checkpoint("Macrophage", "buffer_3"),
        "buffer_4": find_latest_checkpoint("Macrophage", "buffer_4"),
    },
    "K562": {
        "sample_1": find_latest_checkpoint("K562", "sample_1"),
    }
}

INFO - Latest checkpoint for mESC E7.5_rep2: Job 3696113 Epoch 76
INFO - Latest checkpoint for mESC E8.5_rep1: Job 3691937 Epoch 42
INFO - Latest checkpoint for mESC E8.5_rep2: Job 3696124 Epoch 67
INFO - Latest checkpoint for iPSC WT_D13_rep1: Job 3683642 Epoch 75
INFO - Latest checkpoint for Macrophage buffer_1: Job 3685893 Epoch 152
INFO - Latest checkpoint for Macrophage buffer_2: Job 3685903 Epoch 53
INFO - Latest checkpoint for Macrophage buffer_3: Job 3696133 Epoch 249
INFO - Latest checkpoint for Macrophage buffer_4: Job 3696147 Epoch 209
INFO - Latest checkpoint for K562 sample_1: Job 3692409 Epoch 63


In [4]:
species = "mm10"
cell_type = "mESC"
sample_name = "E7.5_rep1"

project_data_dir = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data")

if species == "mm10":
    gene_ref_file = project_data_dir / "genome_data" / "genome_annotation" / "mm10" / "Mus_musculus.GRCm39.115.gtf.gz"
elif species == "hg38":
    gene_ref_file = project_data_dir / "genome_data" / "genome_annotation" / "hg38" / "Homo_sapiens.GRCh38.113.gtf.gz"

genome_fasta_path = project_data_dir / "genome_data" / "reference_genome" / f"{species}" / f"{species}.fa"
chrom_sizes_path = project_data_dir / "genome_data" / "reference_genome" / f"{species}" / f"{species}.chrom.sizes"
chrom_sizes_path = project_data_dir / "genome_data" / "reference_genome" / f"{species}" / f"{species}.chrom.sizes"

train_chroms = [str(i) for i in range(1, 16)]
val_chroms = [ str(i) for i in range(16, 18)]
test_chroms = [str(i) for i in range(18, 20)]

sample_input_data_dir = PROJECT_DIR / "data" / "sample_input_data" / cell_type / sample_name

cell_type_cache_dir = DATA_DIR / f"{cell_type}_cache"

In [47]:
# Load in the ATAC pseudobulk and filter to only include peaks on the test chromosomes
atac_pseudobulk = pd.read_parquet(sample_input_data_dir / "RE_pseudobulk.parquet")
dataset_peaks = atac_pseudobulk.index.to_list()
dataset_peaks = [peak for peak in dataset_peaks if peak.split(":", 1)[0].replace("chr", "") in test_chroms]
logging.info(f"Number of peaks in dataset on test chromosomes: {len(dataset_peaks)}")

# Create a peak to index map for the peaks on the test chromosomes
atac_peak_map = {peak: idx for idx, peak in enumerate(dataset_peaks)}

# Load in the RNA pseudobulk
rna_pseudobulk = pd.read_parquet(sample_input_data_dir / "TG_pseudobulk.parquet")
rna_pseudobulk_norm = rna_pseudobulk.copy()
rna_pseudobulk_norm.index = rna_pseudobulk.index.str.capitalize()

# Load in the peak to gene distance
peak_to_gene_distance = pd.read_parquet(sample_input_data_dir / "peak_to_gene_dist.parquet")
peak_to_gene = peak_to_gene_distance.copy()
peak_to_gene["target_id_norm"] = peak_to_gene["target_id"].str.capitalize()

common_cells = sorted(set(rna_pseudobulk_norm.columns) & set(atac_pseudobulk.columns))

INFO - Number of peaks in dataset on test chromosomes: 12336


In [42]:
print(f"RNA pseudobulk genes: {rna_pseudobulk_norm.index[:5].tolist()}")

RNA pseudobulk genes: ['0610010f05rik', '0610040j01rik', '1010001n08rik', '1190005i06rik', '1600010m07rik']


In [32]:
# Split genes into train/val/test based on chromosome
import gtfparse

def find_genes_by_chromosome(chrom_list: list[str], gene_reference_file: Path):
    gene_ref_df = gtfparse.read_gtf(gene_reference_file, result_type="pandas")
    gene_chrom: pd.DataFrame = gene_ref_df[["seqname", "gene_name"]].rename(
        columns={"seqname": "chrom", "gene_name": "TG"}
    )
    
    matched_genes: np.ndarray = gene_chrom[gene_chrom["chrom"].isin(chrom_list)][
        "TG"
    ].unique()
    return matched_genes

tg_name_array = find_genes_by_chromosome(test_chroms, gene_ref_file)
print(f"Target genes example: {tg_name_array[:10]}")

INFO - Extracted GTF attributes: ['gene_id', 'gene_version', 'gene_name', 'gene_source', 'gene_biotype', 'transcript_id', 'transcript_version', 'transcript_name', 'transcript_source', 'transcript_biotype', 'tag', 'transcript_support_level', 'exon_number', 'exon_id', 'exon_version', 'protein_id', 'protein_version', 'ccds_id']


Target genes example: ['Chst9' 'Gm10036' 'Gm24228' 'Gm4835' 'Gm54353' 'Gm54354' 'Gm7665' 'Cdh2'
 'Gm7670' 'Gm22470']


In [ ]:
# Load the TF and TG name to index mappings from the training cache metadata
with open(cell_type_cache_dir / "tf_tg_training_cache" / sample_name / "metadata.json", "r") as f:
    metadata = json.load(f)
    
tf_name_to_idx = metadata["tf_name_to_idx"]
tg_name_to_idx = metadata["tg_id_to_idx"]

tf_idx_to_name, tg_idx_to_name = create_tf_tg_index_to_name_mappings(metadata)

tf_names = tf_name_to_idx.keys()
tg_names = tg_name_to_idx.keys()

# Subset to only TFs and TGs in the RNA pseudobulk and in the peak-to-gene distance file
tf_names = set(tf_names) & set(rna_pseudobulk_norm.index)
tg_names = set(tg_names) & set(rna_pseudobulk_norm.index) & set(peak_to_gene["target_id_norm"])

print(f"Number of TFs {len(tf_names)} (Example: {list(tf_names)[:5]})")
print(f"Number of TGs {len(tg_names)} (Example: {list(tg_names)[:5]})")

# Create a DataFrame of all TF-TG pairs using MultiIndex
# Uses all TFs with embeddings and all TGs 
pairs = pd.MultiIndex.from_product([tf_names, tg_names], names=["tf_name", "tg_name"])

tf_tg_pairs = pd.DataFrame(index=pairs).reset_index()

print(f"Number of TF-TG pairs: {len(tf_tg_pairs)}")
display(tf_tg_pairs.head())

Number of TFs 137 (Example: ['Klf9', 'Foxc2', 'Myt1l', 'Lmx1b', 'Eomes'])
Number of TGs 1961 (Example: ['Abtb2', 'Flrt3', 'Nuak1', 'Map7', 'Mbnl2'])


In [84]:
def prepare_tftg_lookup_tables(
    peak_to_gene,
    atac_peak_map,
    atac_pseudobulk,
    rna_pseudobulk_norm,
    dataset_peaks,
    common_cells,
    max_precompute_peaks=64,
):
    valid_peak_set = set(atac_peak_map.keys())

    peak_to_gene_valid = peak_to_gene[peak_to_gene["peak_id"].isin(valid_peak_set)].copy()
    peak_to_gene_valid["abs_dist"] = peak_to_gene_valid["TSS_dist"].abs()

    tg_to_peak_info = {}
    for tg_norm, sub in peak_to_gene_valid.groupby("target_id_norm", sort=False):
        sub = sub.sort_values("abs_dist").head(max_precompute_peaks)

        peak_ids = sub["peak_id"].tolist()
        peak_indices = np.asarray([atac_peak_map[p] for p in peak_ids], dtype=np.int64)
        peak_distances = sub["TSS_dist"].to_numpy(dtype=np.float32)

        tg_to_peak_info[tg_norm] = {
            "peak_ids": peak_ids,
            "peak_indices": peak_indices,
            "peak_distances": peak_distances,
        }

    cell_to_idx = {cell: i for i, cell in enumerate(common_cells)}
    atac_mat = (
        atac_pseudobulk
        .reindex(index=dataset_peaks, columns=common_cells)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )
    rna_mat = (
        rna_pseudobulk_norm
        .reindex(columns=common_cells)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )
    gene_to_rna_idx = {gene: i for i, gene in enumerate(rna_pseudobulk_norm.index)}

    return tg_to_peak_info, cell_to_idx, atac_mat, rna_mat, gene_to_rna_idx

In [ ]:
def build_tftg_inputs(
    tf_tg_df,
    max_peaks_per_tg=64,
    max_cells_per_pair=8,
    seed=123,
    silence=False,
    *,
    tg_to_peak_info,
    cell_to_idx,
    atac_mat,
    rna_mat,
    gene_to_rna_idx,
    common_cells,
    tf_name_to_idx,
    tg_name_to_idx,
):
    """
    Build one compact item per TF-TG edge.

    Output shapes:
        tf_idx:             [E]
        tg_idx:             [E]
        peak_indices:       [E, P]
        peak_distance:      [E, P]
        peak_mask:          [E, P]
        peak_accessibility: [E, C, P]
        tf_expression:      [E, C]
        tg_expression:      [E, C]
    """

    rng = np.random.default_rng(seed)

    tf_names = []
    tg_names = []
    cell_ids_all = []

    tf_indices = []
    tg_indices = []
    peak_indices_all = []
    peak_access_all = []
    peak_dist_all = []
    peak_masks_all = []
    tf_expr_all = []
    tg_expr_all = []

    common_cells = list(common_cells)
    n_common_cells = len(common_cells)

    
    num_skipped = 0
    
    tgs_in_peak_info = set(tg_to_peak_info.keys())
    
    print(f"Total TF-TG pairs before filtering: {len(tf_tg_df)}")
    print(f"TGs missing from tg_to_peak_info: {len(set(tf_tg_df['tg_name'].unique()) - tgs_in_peak_info)}")
    print(f"TFs missing from tf_name_to_idx: {len(set(tf_tg_df['tf_name'].unique()) - set(tf_name_to_idx.keys()))}")
    print(f"TGs missing from tg_name_to_idx: {len(set(tf_tg_df['tg_name'].unique()) - set(tg_name_to_idx.keys()))}")
    
    tf_tg_df = tf_tg_df[
        (tf_tg_df["tg_name"].isin(set(tg_to_peak_info.keys()))) &
        (tf_tg_df["tf_name"].isin(set(tf_name_to_idx.keys()))) &
        (tf_tg_df["tg_name"].isin(set(tg_name_to_idx.keys())))
        ].copy()
    
    print(f"Number of TF-TG pairs after filtering for valid TFs, TGs, and TGs with peak info: {len(tf_tg_df)}")
    
    n_total = len(tf_tg_df)
    log_every = max(1, n_total // 50)

    for i, row in enumerate(tf_tg_df.itertuples(index=False), start=1):
        if silence == False:
            if i == 1 or i % log_every == 0 or i == n_total:
                logging.info(f"Building compact TF-TG edges: {100 * i / n_total:.1f}% ({i-num_skipped:,}/{n_total:,}) (skipped {num_skipped:,})")

        tf_name = row.tf_name
        tg_name = row.tg_name
        
        tf_idx = tf_name_to_idx.get(tf_name)
        tg_idx = tg_name_to_idx.get(tg_name)
        
        if tf_idx is None or tg_idx is None:
            num_skipped += 1
            continue

        peak_info = tg_to_peak_info.get(tg_name)
        if peak_info is None:
            num_skipped += 1
            continue

        peak_indices_real = list(peak_info["peak_indices"][:max_peaks_per_tg])
        peak_dst_real = list(peak_info["peak_distances"][:max_peaks_per_tg])

        n_peaks = len(peak_indices_real)
        if n_peaks == 0:
            num_skipped += 1
            continue

        peak_indices = np.asarray(peak_indices_real, dtype=np.int64)
        peak_dst = np.asarray(peak_dst_real, dtype=np.float32)
        peak_mask = np.ones(n_peaks, dtype=bool)

        if n_peaks < max_peaks_per_tg:
            pad_len = max_peaks_per_tg - n_peaks

            peak_indices = np.pad(
                peak_indices,
                (0, pad_len),
                constant_values=0,
            )

            peak_dst = np.pad(
                peak_dst,
                (0, pad_len),
                constant_values=0.0,
            )

            peak_mask = np.pad(
                peak_mask,
                (0, pad_len),
                constant_values=False,
            )

        # Sample cells
        if max_cells_per_pair is None or max_cells_per_pair >= n_common_cells:
            sampled_cells = common_cells
        else:
            sampled_cells = rng.choice(
                common_cells,
                size=max_cells_per_pair,
                replace=False,
            ).tolist()

        sampled_cell_indices = np.asarray(
            [cell_to_idx[c] for c in sampled_cells],
            dtype=np.int64,
        )

        C = len(sampled_cell_indices)
        P = max_peaks_per_tg

        # ATAC accessibility: [C, P]
        peak_acc_matrix = np.zeros((C, P), dtype=np.float32)
        peak_acc_matrix[:, :n_peaks] = atac_mat[
            np.ix_(peak_indices_real, sampled_cell_indices)
        ].T

        # RNA expression: [C]
        tf_rna_idx = gene_to_rna_idx.get(tf_name)
        tg_rna_idx = gene_to_rna_idx.get(tg_name)

        if tf_rna_idx is None or tg_rna_idx is None:
            num_skipped += 1
            continue
        
        tf_expr_vals = np.asarray(
            rna_mat[tf_rna_idx, sampled_cell_indices],
            dtype=np.float32,
        ).reshape(-1)

        tg_expr_vals = np.asarray(
            rna_mat[tg_rna_idx, sampled_cell_indices],
            dtype=np.float32,
        ).reshape(-1)

        # Append once per TF-TG edge
        tf_names.append(tf_name)
        tg_names.append(tg_name)
        cell_ids_all.append(sampled_cells)

        tf_indices.append(tf_idx)
        tg_indices.append(tg_idx)
        peak_indices_all.append(peak_indices)
        peak_access_all.append(peak_acc_matrix)
        peak_dist_all.append(peak_dst)
        peak_masks_all.append(peak_mask)
        tf_expr_all.append(tf_expr_vals)
        tg_expr_all.append(tg_expr_vals)

    return {
        "tf_name": tf_names,
        "tg_name": tg_names,
        "cell_ids": cell_ids_all,

        "tf_idx": torch.tensor(tf_indices, dtype=torch.long),
        "tg_idx": torch.tensor(tg_indices, dtype=torch.long),

        "peak_indices": torch.tensor(np.stack(peak_indices_all), dtype=torch.long),
        "peak_accessibility": torch.tensor(np.stack(peak_access_all), dtype=torch.float32),
        "peak_mask": torch.tensor(np.stack(peak_masks_all), dtype=torch.bool),
        "peak_distance": torch.tensor(np.stack(peak_dist_all), dtype=torch.float32),

        "tf_expression": torch.tensor(np.stack(tf_expr_all), dtype=torch.float32),
        "tg_expression": torch.tensor(np.stack(tg_expr_all), dtype=torch.float32),
    }

In [86]:
# Create the centered one-hot encoded ATAC peak array for the test set
logging.info("Creating centered one-hot encoded ATAC peak array for the test set...")
atac_peak_array = utils.create_centered_peak_onehot_array(
    peak_ids=dataset_peaks,
    genome_fasta=genome_fasta_path,
    chrom_sizes=utils.load_chrom_sizes(chrom_sizes_path),
    peak_id_to_idx=atac_peak_map,
    flank_size=128,
    dtype=np.uint8,
    pad_out_of_bounds=True,
    num_workers=10,
    show_progress=False,
    chunk_size=10000,
)
atac_peak_tensor = torch.as_tensor(atac_peak_array, dtype=torch.uint8).float()
print(f"ATAC peak tensor shape: {atac_peak_tensor.shape}")

INFO - Creating centered one-hot encoded ATAC peak array for the test set...


ATAC peak tensor shape: torch.Size([12336, 256, 4])


In [87]:

logging.info(f"Constructing TF-TG lookup tables for test set evaluation")
tg_to_peak_info, cell_to_idx, atac_mat, rna_mat, gene_to_rna_idx = prepare_tftg_lookup_tables(
    peak_to_gene=peak_to_gene,
    atac_peak_map=atac_peak_map,
    atac_pseudobulk=atac_pseudobulk,
    rna_pseudobulk_norm=rna_pseudobulk_norm,
    dataset_peaks=dataset_peaks,
    common_cells=common_cells,
    max_precompute_peaks=8,
)
print(f"Number of TGs with at least one peak: {len(tg_to_peak_info)}")
print(f"Example TG to peak info entry: {next(iter(tg_to_peak_info.items()))}")
print(f"Number of common cells: {len(common_cells)}")
print(f"ATAC matrix shape: {atac_mat.shape}, RNA matrix shape: {rna_mat.shape}")

INFO - Constructing TF-TG lookup tables for test set evaluation


Number of TGs with at least one peak: 2923
Example TG to peak info entry: ('Rasgrp2', {'peak_ids': ['chr19:6448513-6449370', 'chr19:6451943-6452817', 'chr19:6462686-6463473'], 'peak_indices': array([7144, 7145, 7146]), 'peak_distances': array([   0.,  131., 5038.], dtype=float32)})
Number of common cells: 5199
ATAC matrix shape: (12336, 5199), RNA matrix shape: (2453, 5199)


In [91]:
# Build the compact TF-TG input dataset for the test set
common_build_kwargs = dict(
    max_peaks_per_tg=8,
    max_cells_per_pair=16,
    tg_to_peak_info=tg_to_peak_info,
    cell_to_idx=cell_to_idx,
    atac_mat=atac_mat,
    rna_mat=rna_mat,
    gene_to_rna_idx=gene_to_rna_idx,
    common_cells=common_cells,
    tf_name_to_idx=tf_name_to_idx,
    tg_name_to_idx=tg_name_to_idx,
)

logging.info("Building TF-TG input dataset...")
tftg_inputs_test = build_tftg_inputs(
    tf_tg_pairs,
    seed=125,
    silence=False,
    **common_build_kwargs,
)

# Load the lookup tensors
tf_embeddings_tensor = torch.load(
    cell_type_cache_dir / "tf_embeddings.pt",
    weights_only=True,
)
tf_mask_tensor = torch.load(
    cell_type_cache_dir / "tf_masks.pt",
    weights_only=True,
)

INFO - Building TF-TG input dataset...
INFO - Building compact TF-TG edges: 0.0% (1/18,906) (skipped 0)


Total TF-TG pairs before filtering: 268657
TGs missing from tg_to_peak_info: 1823
TFs missing from tf_name_to_idx: 0
TGs missing from tg_name_to_idx: 0
Number of TF-TG pairs after filtering for valid TFs, TGs, and TGs with peak info: 18906


INFO - Building compact TF-TG edges: 2.0% (378/18,906) (skipped 0)
INFO - Building compact TF-TG edges: 4.0% (756/18,906) (skipped 0)
INFO - Building compact TF-TG edges: 6.0% (1,134/18,906) (skipped 0)
INFO - Building compact TF-TG edges: 8.0% (1,512/18,906) (skipped 0)
INFO - Building compact TF-TG edges: 10.0% (1,890/18,906) (skipped 0)
INFO - Building compact TF-TG edges: 12.0% (2,268/18,906) (skipped 0)
INFO - Building compact TF-TG edges: 14.0% (2,646/18,906) (skipped 0)
INFO - Building compact TF-TG edges: 16.0% (3,024/18,906) (skipped 0)
INFO - Building compact TF-TG edges: 18.0% (3,402/18,906) (skipped 0)
INFO - Building compact TF-TG edges: 20.0% (3,780/18,906) (skipped 0)
INFO - Building compact TF-TG edges: 22.0% (4,158/18,906) (skipped 0)
INFO - Building compact TF-TG edges: 24.0% (4,536/18,906) (skipped 0)
INFO - Building compact TF-TG edges: 26.0% (4,914/18,906) (skipped 0)
INFO - Building compact TF-TG edges: 28.0% (5,292/18,906) (skipped 0)
INFO - Building compact TF-T

In [95]:
import torch
from torch.utils.data import Dataset

class TFTGEdgeBagDataset(Dataset):
    def __init__(
        self,
        inputs,
        *,
        tf_embeddings_tensor,
        tf_mask_tensor,
        atac_peak_tensor,
    ):
        self.inputs = inputs
        self.tf_embeddings_tensor = tf_embeddings_tensor
        self.tf_mask_tensor = tf_mask_tensor
        self.atac_peak_tensor = atac_peak_tensor

    def __len__(self):
        return len(self.inputs["tf_idx"])

    def __getitem__(self, idx):
        tf_idx = self.inputs["tf_idx"][idx]
        tg_idx = self.inputs["tg_idx"][idx]

        peak_indices = self.inputs["peak_indices"][idx]          # [P]
        peak_sequences = self.atac_peak_tensor[peak_indices]     # [P, L, 4]

        tf_embedding = self.tf_embeddings_tensor[tf_idx]         # [T, D]
        tf_mask = self.tf_mask_tensor[tf_idx]                    # [T]

        return {
            "tf_name": self.inputs["tf_name"][idx],
            "tg_name": self.inputs["tg_name"][idx],
            "cell_ids": self.inputs["cell_ids"][idx],
            "tf_idx": tf_idx,
            "tg_idx": tg_idx,
            "tf_embedding": tf_embedding.float(),
            "tf_mask": tf_mask.bool(),
            "peak_indices": peak_indices,
            "peak_sequences": peak_sequences,
            "peak_distance": self.inputs["peak_distance"][idx].float(),
            "peak_mask": self.inputs["peak_mask"][idx].bool(),
            "peak_accessibility": self.inputs["peak_accessibility"][idx].float(),
            "tf_expression": self.inputs["tf_expression"][idx].float(),
            "tg_expression": self.inputs["tg_expression"][idx].float(),
        }
        
def collate_tftg_edge_bags(batch):
    output = {
        "tf_idx": torch.stack([b["tf_idx"] for b in batch]).long(),
        "tg_idx": torch.stack([b["tg_idx"] for b in batch]).long(),

        "tf_embedding": torch.stack([b["tf_embedding"] for b in batch]),
        "tf_mask": torch.stack([b["tf_mask"] for b in batch]),

        "peak_indices": torch.stack([b["peak_indices"] for b in batch]),
        "peak_sequences": torch.stack([b["peak_sequences"] for b in batch]),
        "peak_distance": torch.stack([b["peak_distance"] for b in batch]),
        "peak_mask": torch.stack([b["peak_mask"] for b in batch]),

        "peak_accessibility": torch.stack([b["peak_accessibility"] for b in batch]),
        "tf_expression": torch.stack([b["tf_expression"] for b in batch]),
        "tg_expression": torch.stack([b["tg_expression"] for b in batch]),

        "tf_name": [b["tf_name"] for b in batch],
        "tg_name": [b["tg_name"] for b in batch],
        "cell_ids": [b["cell_ids"] for b in batch],
    }

    E, C = output["tf_expression"].shape
    output["cell_mask"] = torch.ones(E, C, dtype=torch.bool)

    return output

In [97]:
# Create the dataset for the test set using the loaded inputs and lookup tensors
dataset = TFTGEdgeBagDataset(
    tftg_inputs_test,
    tf_embeddings_tensor=tf_embeddings_tensor,
    tf_mask_tensor=tf_mask_tensor,
    atac_peak_tensor=atac_peak_tensor
)

# Create the DataLoader for the test set
num_workers = 8
loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=(num_workers > 0),
    prefetch_factor=2 if num_workers > 0 else None,
    collate_fn=collate_tftg_edge_bags,
)

print(f"Created DataLoader with {len(dataset)} samples and batch size {loader.batch_size}")

Created DataLoader with 18906 samples and batch size 8


In [100]:
tf_dna_model_chkpt = tf_dna_model_checkpoints[cell_type]
tf_tg_model_chkpt = tf_tg_model_checkpoints[cell_type][sample_name]

# Load the TF→TG model
tf_tg_model = load_tf_tg_regulation_model(
    tf_dna_model_chkpt, 
    tf_tg_model_chkpt, 
    tf_embeddings_tensor, 
    tf_mask_tensor
    )

# Generate the model predictions for the test set and create a DataFrame with TF names, TG names, and predicted scores
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
prediction_df = generate_model_predictions(tf_tg_model.model, loader, device, tf_idx_to_name, tg_idx_to_name)

Using device: cuda


Evaluating: 100%|███████████████████████████████████████████████| 2364/2364 [01:44<00:00, 22.66it/s]


In [101]:
prediction_df.head()

,Source,Target,Score
0,Ar,4930484i04rik,0.412394
1,Ar,9630014m24rik,0.595664
2,Ar,A330040f15rik,0.448759
3,Ar,A930007i19rik,0.541772
4,Ar,Abcc2,0.630899
